# Case Study 1: Hospital Readmission Prediction

**Objective:** Predict whether a patient will be readmitted within 30 days after hospital discharge using **Logistic Regression with L2 regularization**.

### Required tasks
- Load a hospital readmission dataset.
- Perform basic data inspection and preprocessing.
- Use patient clinical/demographic features as predictors.
- Train Logistic Regression with **L2 regularization**.
- Evaluate using **ROC-AUC**.
- Inspect the confusion matrix and discuss the clinical cost of false negatives vs. false positives.

> This notebook uses the Kaggle synthetic hospital readmission dataset specified for this case study. The dataset contains a binary 30-day readmission target (`readmitted_30_days`) and clinical/hospital-related variables.


In [ ]:
# Install/import required libraries
!pip -q install kagglehub

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


## 1. Download the Kaggle dataset

In [ ]:
import kagglehub

# Download the dataset
path = kagglehub.dataset_download(
    "siddharth0935/hospital-readmission-predictionsynthetic-dataset"
)

print("Path to dataset files:", path)
print("Files:", os.listdir(path))


In [ ]:
# Find CSV files automatically
csv_files = glob.glob(os.path.join(path, "*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV file was found in the downloaded dataset folder.")

print("CSV files found:")
for f in csv_files:
    print(" -", f)

# Load the first CSV file
df = pd.read_csv(csv_files[0])

print("\nDataset shape:", df.shape)
display(df.head())


## 2. Understand the dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))


In [ ]:
# Identify the 30-day readmission target
target_candidates = [
    "readmitted_30_days",
    "readmitted",
    "Readmitted_30_Days",
    "Readmitted"
]

target = next((c for c in target_candidates if c in df.columns), None)

if target is None:
    raise ValueError(
        "Could not find the 30-day readmission target. "
        f"Available columns are: {df.columns.tolist()}"
    )

print("Target column:", target)
print("\nTarget distribution:")
display(df[target].value_counts(dropna=False))


## 3. Basic exploratory analysis

We first inspect the class balance. In a readmission problem, the positive class represents patients who are readmitted within 30 days.


In [ ]:
# Plot target distribution
plt.figure(figsize=(6, 4))
df[target].value_counts().plot(kind="bar")
plt.title("30-Day Readmission Distribution")
plt.xlabel("Readmission Status")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Readmission rate:")
print((df[target].value_counts(normalize=True) * 100).round(2))


## 4. Prepare the target variable

In [ ]:
# Convert common Yes/No representations into 0/1
if df[target].dtype == "object":
    target_values = df[target].astype(str).str.strip().str.lower()
    mapping = {
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    }
    y = target_values.map(mapping)
else:
    y = pd.to_numeric(df[target], errors="coerce")

if y.isna().any():
    raise ValueError(
        "Some target values could not be converted to 0/1. "
        f"Unique target values: {df[target].unique()}"
    )

y = y.astype(int)

print("Target encoded as:")
print("0 = Not readmitted within 30 days")
print("1 = Readmitted within 30 days")
print("\nEncoded distribution:")
print(y.value_counts())


## 5. Prepare features

We remove the target from the predictors. An obvious identifier column is also excluded when present because an ID should not be used as a clinical predictor.

The preprocessing pipeline:
- Numerical variables → median imputation + standardization.
- Categorical variables → most-frequent imputation + one-hot encoding.


In [ ]:
X = df.drop(columns=[target]).copy()

# Remove obvious identifier columns if they exist
id_like = [
    c for c in X.columns
    if c.lower() in {"id", "patient_id", "patientid", "record_id", "recordid"}
]

if id_like:
    X = X.drop(columns=id_like)
    print("Removed identifier columns:", id_like)

# Convert blood pressure strings such as "120/80" into two numeric features
bp_candidates = [c for c in X.columns if c.lower() in {"blood_pressure", "blood pressure"}]

for bp_col in bp_candidates:
    bp_split = X[bp_col].astype(str).str.extract(r"^(\d+(?:\.\d+)?)[/]\s*(\d+(?:\.\d+)?)$")
    X["systolic_bp"] = pd.to_numeric(bp_split[0], errors="coerce")
    X["diastolic_bp"] = pd.to_numeric(bp_split[1], errors="coerce")
    X = X.drop(columns=[bp_col])
    print(f"Converted {bp_col} into systolic_bp and diastolic_bp.")

print("Final feature columns:")
print(X.columns.tolist())
print("\nFeature matrix shape:", X.shape)


In [ ]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)
print("Training positive rate:", round(y_train.mean() * 100, 2), "%")
print("Testing positive rate :", round(y_test.mean() * 100, 2), "%")


## 6. Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## 7. Logistic Regression with L2 regularization

L2 regularization is explicitly selected using:

`penalty="l2"`

The regularization strength is controlled by `C`. A smaller `C` means stronger regularization. Here we use `C=1.0` as a standard starting point.


In [ ]:
logistic_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=2000,
    solver="liblinear",
    random_state=42
)

model = Pipeline([
    ("preprocessing", preprocessor),
    ("logistic_regression", logistic_model)
])

# Train
model.fit(X_train, y_train)

print("Logistic Regression with L2 regularization trained successfully.")


## 8. Evaluate the model

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Main metrics
roc_auc = roc_auc_score(y_test, y_proba)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Readmitted", "Readmitted"],
    zero_division=0
))


## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print("\nTN (True Negative) :", tn)
print("FP (False Positive):", fp)
print("FN (False Negative):", fn)
print("TP (True Positive) :", tp)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Not Readmitted", "Readmitted"])
plt.yticks([0, 1], ["Not Readmitted", "Readmitted"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


## 10. ROC curve

ROC-AUC measures how well the model separates patients who are readmitted within 30 days from those who are not, across different classification thresholds.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - 30-Day Hospital Readmission")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 11. Clinical cost: False Negatives vs. False Positives

### False Negative (FN)
The model predicts **not readmitted**, but the patient is actually readmitted within 30 days.

This can be clinically costly because a high-risk patient may not receive:
- additional follow-up,
- medication review,
- discharge planning,
- early outpatient monitoring, or
- other preventive interventions.

### False Positive (FP)
The model predicts **readmission risk**, but the patient is not actually readmitted.

This can lead to:
- additional monitoring,
- extra follow-up appointments,
- unnecessary use of clinical resources,
- increased workload for healthcare staff.

### Which error is usually more concerning?

For a readmission-risk screening system, **false negatives can be more costly than false positives** because missing a genuinely high-risk patient may mean missing an opportunity for intervention.

Therefore, a hospital may choose a classification threshold below 0.50 if the goal is to increase recall/sensitivity and reduce false negatives. The exact threshold should be selected using clinical and operational cost estimates rather than accuracy alone.


## 12. Optional threshold analysis

In [ ]:
# Compare several probability thresholds
threshold_results = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred_threshold = (y_proba >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, pred_threshold).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Recall": recall_score(y_test, pred_threshold, zero_division=0),
        "Precision": precision_score(y_test, pred_threshold, zero_division=0),
        "False Positives": fp_t,
        "False Negatives": fn_t,
        "True Positives": tp_t,
        "True Negatives": tn_t
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(3))


## 13. Model interpretation

Logistic Regression coefficients indicate the direction of association with predicted readmission risk after preprocessing.

Positive coefficients push the prediction toward readmission; negative coefficients push it toward non-readmission. Because numerical features are standardized, their coefficients are more directly comparable in magnitude than they would be on their original scales.


In [ ]:
# Extract feature names and coefficients
preprocessor_fitted = model.named_steps["preprocessing"]
lr_fitted = model.named_steps["logistic_regression"]

feature_names = preprocessor_fitted.get_feature_names_out()
coefficients = lr_fitted.coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Absolute_Coefficient", ascending=False)

print("Top features by absolute Logistic Regression coefficient:")
display(coef_df.head(15))


# Final Conclusion

This case study used **Logistic Regression with L2 regularization** to predict **30-day hospital readmission** from patient-level clinical and hospital-related information.

The workflow included data loading, target encoding, train/test splitting, missing-value handling, categorical encoding, numerical standardization, and model training.

The model was evaluated primarily using **ROC-AUC**, along with precision, recall, F1-score, accuracy, and the confusion matrix.

In this clinical application, **false negatives are particularly important** because a patient incorrectly classified as low-risk could miss additional monitoring or intervention. False positives also have a cost because they can consume healthcare resources unnecessarily. Therefore, the classification threshold should ultimately reflect the relative clinical and operational cost of these two types of errors.

> **Important:** Run all cells in Google Colab before reporting the numerical ROC-AUC, confusion-matrix counts, or other metric values in your submission.
